# 03. Lecke - Ügynöki Tervezési Minták

Ebben a leckében három alapvető tervezési mintát vizsgálunk, amelyek hatékony MI ügynökök építéséhez szükségesek:

1. **Világos ügynök utasítások** — Precíz, szerepet meghatározó kérések kidolgozása az ügynök viselkedésének irányítására  
2. **Strukturált kimenet Pydantic modellekkel** — Annak biztosítása, hogy az ügynökök kiszámítható, validált adatokat adjanak vissza  
3. **Egyszeres felelősségű ügynökök** — Olyan fókuszált ügynökök tervezése, amelyek jól végzik el az egy feladatot  

Minden mintát egy **utazási célpont ajánló** példán keresztül alkalmazunk, fokozatosan építve egy olyan rendszert, amely képes célpontokat javasolni, ellenőrizni az elérhetőséget és kezelni a logisztikát.


## Beállítás


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## 1. Minta: Egyértelmű ügynöki utasítások

A leghatásosabb minta egyben a legegyszerűbb is: egyértelmű, részletes utasításokat írni az ügynököd számára.

A jó utasítások meghatározzák:
- **Ki** az az ügynök (személyiség és hangnem)
- **Mit** kell tennie (lépésről lépésre a feladatok)
- **Hogyan** kell viselkednie (korlátok és stílus)

Az alábbiakban létrehozunk egy utazási concierge ügynököt kifejezett utasításokkal, amelyek alakítják minden általa adott választ.


In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Minta 2: Strukturált kimenet Pydantic modellekkel

A szabad szöveg hasznos a beszélgetéshez, de az alrendszereknek strukturált adatokra van szükségük.
A **Pydantic modellek** és egy **eszközfüggvény** párosításával képesek vagyunk:

- Pontos sémát meghatározni az ügynök kimenetéhez
- Automatikusan validálni a válaszokat
- Megbízhatóan integrálni az ügynök eredményeit az alkalmazás logikájába

Bemutatunk egy eszközt is, amely visszaadja az úti cél részleteit, így az ügynök valós adatok alapján tudja megalapozni ajánlásait.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = await provider.create_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## 3. Minta: Egyetlen felelősségű ügynökök

Az összetett feladatok előnyösek, ha a munkát több, egyetlen felelősséggel bíró, fókuszált ügynökre osztjuk szét:

- Egy **Úti cél szakértő**, aki ismeri a helyeket és a rendelkezésre állást
- Egy **Logisztikai tervező**, aki a járatokat, hoteleket és útitervet kezeli

Ez tükrözi a szoftverfejlesztés elvét, az *érdekkülönválasztást* — minden ügynök könnyebben tesztelhető, karbantartható és fejleszthető önállóan.


In [ ]:
destination_agent = await provider.create_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = await provider.create_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Összefoglalás

Ebben a leckében három ügynöktervezési mintát alkalmaztunk egy utazási ajánló szcenárióban:

| Minta | Kulcsötlet | Előny |
|---|---|---|
| **Egyértelmű utasítások** | Definiáljuk az személyiséget, felelősségeket és korlátokat előre | Következetes, márkának megfelelő ügynök viselkedés |
| **Strukturált kimenet** | Használjunk Pydantic modelleket válaszformátumként | Ellenőrzött, gépileg olvasható eredmények |
| **Egyszeri felelősség** | Adjunk minden ügynöknek egy fókuszált feladatot | Könnyebb tesztelés, karbantartás és összetétel |

Ezek a minták természetesen összehangolhatók — egyértelmű utasításokat kombinálhatunk strukturált kimenettel egy egyszeri felelősségű ügynökön belül, hogy robusztus, élesben használható rendszereket építsünk.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Felelősségkizárás**:  
Ez a dokumentum az AI fordító szolgáltatás, a [Co-op Translator](https://github.com/Azure/co-op-translator) segítségével készült. Bár a pontosságra törekszünk, kérjük, vegye figyelembe, hogy az automatikus fordítások hibákat vagy pontatlanságokat tartalmazhatnak. A dokumentum eredeti, anyanyelvi változatát kell tekinteni a hivatalos forrásnak. Fontos információk esetén javasolt szakmai, emberi fordítás igénybevétele. Nem vállalunk felelősséget a fordítás használatából eredő esetleges félreértésekért vagy téves értelmezésekért.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
